前置宣告与设定路经

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

DATA_DIR = "../data/raw/"
OUTPUT_DIR = "../data/processed/"

导入文档

In [2]:
population = pd.read_excel(DATA_DIR + "人口规模.xlsx")
age = pd.read_excel(DATA_DIR + "年龄结构.xlsx")
urban = pd.read_excel(DATA_DIR + "城镇化率.xlsx")
density = pd.read_excel(DATA_DIR + "人口密度.xlsx")

# print("人口规模：", population.shape)
# print("年龄结构：", age.shape)
# print("城镇化率：", urban.shape)
# print("人口密度：", density.shape)

规范化所有数据

In [3]:
population = population.rename(columns={
    "城市名称": "city_id",
    "年份": "year",
    "常住人口（万人）": "population",
    "户籍人口（万人）": "registered_population"
})

# population.head()

age = age.rename(columns={
    "城市名称": "city_id",
    "年份": "year",
    "0-14": "age_0_14",
    "15-64": "age_15_64",
    "65+": "age_65_plus"
})

urban = urban.rename(columns={
    "城市名称": "city_id",
    "年份": "year",
    "urbanizationRate": "urbanization_rate"
})

density = density.rename(columns={
    "城市名称": "city_id",
    "年份": "year",
    "人口密度（人/平方公里）": "population_density"
})


统一数据类型

In [4]:
dfs = [population, age, urban, density]

for df in dfs:
    df["city_id"] = df["city_id"].astype(str)
    df["year"] = pd.to_numeric(df["year"], errors="coerce")

for df in dfs:
    print(df.dtypes)
    print()

city_id                      str
year                       int64
population               float64
registered_population    float64
dtype: object

city_id            str
year             int64
age_0_14         int64
age_15_64      float64
age_65_plus    float64
dtype: object

city_id                str
year                 int64
urbanization_rate    int64
dtype: object

year                  int64
city_id                 str
population_density    int64
dtype: object



In [5]:
def check_duplicate_key(df, name):
    duplicated = df[df.duplicated(
        subset=["city_id", "year"],
        keep=False
    )]

    print(f"{name} 重复主键行数：{len(duplicated)}")

    if len(duplicated) > 0:
        display(
            duplicated.sort_values(["city_id", "year"])
        )
        
        
check_duplicate_key(population, "人口规模")
check_duplicate_key(age, "年龄结构")
check_duplicate_key(urban, "城镇化率")
check_duplicate_key(density, "人口密度")


人口规模 重复主键行数：0
年龄结构 重复主键行数：0
城镇化率 重复主键行数：0
人口密度 重复主键行数：2


,year,city_id,population_density
394,2002,city31,2315
395,2002,city31,422


人口密度有问题，删除所有“不满足 city31 + 2002 + 2315”的行。

In [6]:
density[
    (density["city_id"] == "city31") &
    (density["year"].between(2000, 2005))
].sort_values("year")

density = density[
    ~(
        (density["city_id"] == "city31") &
        (density["year"] == 2002) &
        (density["population_density"] == 2315)
    )
].copy()

检查缺失值

In [7]:
def check_missing(df, name):
    print("=" * 50)
    print(name)
    print(df.isna().sum())
    
check_missing(population, "人口规模")
check_missing(age, "年龄结构")
check_missing(urban, "城镇化率")
check_missing(density, "人口密度")

人口规模
city_id                   0
year                      0
population               20
registered_population     6
dtype: int64
年龄结构
city_id         0
year            0
age_0_14        0
age_15_64      20
age_65_plus    15
dtype: int64
城镇化率
city_id              0
year                 0
urbanization_rate    0
dtype: int64
人口密度
year                  0
city_id               0
population_density    0
dtype: int64


排序年份

In [8]:
population = population.sort_values(
    ["city_id", "year"]
).reset_index(drop=True)

age = age.sort_values(
    ["city_id", "year"]
).reset_index(drop=True)

urban = urban.sort_values(
    ["city_id", "year"]
).reset_index(drop=True)

density = density.sort_values(
    ["city_id", "year"]
).reset_index(drop=True)

最终检查数据清洗的部分（此部分输出加入ppt）

In [9]:
def quality_report(df, name):
    print("=" * 60)
    print(name)

    print("Shape：", df.shape)
    print("城市数量：", df["city_id"].nunique())
    print("最早年份：", df["year"].min())
    print("最晚年份：", df["year"].max())

    print(
        "重复 city-year：",
        df.duplicated(["city_id", "year"]).sum()
    )

    print("缺失值总数：", df.isna().sum().sum())
    
quality_report(population, "人口规模")
quality_report(age, "年龄结构")
quality_report(urban, "城镇化率")
quality_report(density, "人口密度")

人口规模
Shape： (717, 4)
城市数量： 40
最早年份： 2001
最晚年份： 2021
重复 city-year： 0
缺失值总数： 26
年龄结构
Shape： (118, 5)
城市数量： 40
最早年份： 2000
最晚年份： 2020
重复 city-year： 0
缺失值总数： 35
城镇化率
Shape： (675, 3)
城市数量： 40
最早年份： 2003
最晚年份： 2022
重复 city-year： 0
缺失值总数： 0
人口密度
Shape： (757, 3)
城市数量： 40
最早年份： 2000
最晚年份： 2022
重复 city-year： 0
缺失值总数： 0


保存

In [10]:
population.to_csv(
    OUTPUT_DIR + "population_clean.csv",
    index=False
)

age.to_csv(
    OUTPUT_DIR + "age_clean.csv",
    index=False
)

urban.to_csv(
    OUTPUT_DIR + "urban_clean.csv",
    index=False
)

density.to_csv(
    OUTPUT_DIR + "density_clean.csv",
    index=False
)